# IVS Weekmonitor Analysis & UN/LOCODE Mapping

This notebook demonstrates how to process the IVS weekmonitor data from Rijkswaterstaat, aggregate cargo by origin-destination (OD) pairs, and map these flows to the inland waterway network using ISRS locodes and Zenodo reference data.

In [ ]:
import pathlib
import zipfile
import io
import requests
import pickle
import itertools
import logging

import pandas as pd
import geopandas as gpd
import shapely.wkt
import shapely.geometry
import networkx as nx
import matplotlib.pyplot as plt

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

## 1. Load IVS Data Directly from Internet
We fetch the latest weekmonitor ZIP and read the CSV content in-memory.

In [ ]:
ivs_url = "https://downloads.rijkswaterstaatdata.nl/scheepvaart/goederenvervoer/archief/IVS_weekmonitor_31MAR2026_20260401_051939.zip"

logger.info(f"Fetching IVS data from {ivs_url}...")
r = requests.get(ivs_url)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_filename = z.namelist()[0]
    with z.open(csv_filename) as f:
        ivs_df = pd.read_csv(f, sep=";")

logger.info(f"Loaded {len(ivs_df)} records from {csv_filename}.")
ivs_df.head()

## 2. Aggregate OD Data
We group the cargo data by origin (`UNLO_herkomst`) and destination (`UNLO_bestemming`).

In [ ]:
# Aggregate transported weight (v38_Vervoerd_gewicht) by OD pair
od_pairs = (
    ivs_df.groupby(["UNLO_herkomst", "UNLO_bestemming"])["v38_Vervoerd_gewicht"]
    .sum()
    .reset_index()
)
od_pairs = od_pairs.sort_values(by="v38_Vervoerd_gewicht", ascending=False)

logger.info("Top 5 OD Pairs by Weight:")
for idx, row in od_pairs.head(5).iterrows():
    logger.info(
        f"  {row['UNLO_herkomst']} -> {row['UNLO_bestemming']}: {row['v38_Vervoerd_gewicht']:,} kg"
    )

## 3. Load Network and Map UN/LOCODEs
We load the merged graph and map the 5-character UN/LOCODEs to network nodes (specifically `sectionjunction` types).

In [ ]:
graph_path = pathlib.Path("../output/merged-graph/graph.pickle")
with graph_path.open("rb") as f:
    graph = pickle.load(f)

nodes_gdf = gpd.GeoDataFrame(graph.nodes.values(), index=graph.nodes.keys())
logger.info(f"Loaded network with {len(nodes_gdf)} nodes.")


def find_nodes_for_locode(locode):
    """Find network nodes matching the given 5-char UN/LOCODE, focusing on sectionjunctions."""
    # Match by prefix or inclusion in ISRS locode
    mask = nodes_gdf["locode"].str.startswith(locode, na=False) | nodes_gdf[
        "locode"
    ].str.contains(locode[2:], na=False)

    # Focus on sectionjunction nodes
    junction_mask = nodes_gdf["geo_type"] == "sectionjunction"

    nodes = nodes_gdf[mask & junction_mask]
    assert not nodes.empty, f"No sectionjunction nodes found for {locode}"
    return nodes


# Example: Rotterdam (NLRTM) to Antwerp (BEANR)
origin_locode = "NLRTM"
dest_locode = "BEANR"

origin_nodes = find_nodes_for_locode(origin_locode)
dest_nodes = find_nodes_for_locode(dest_locode)

logger.info(f"Found {len(origin_nodes)} sectionjunction nodes for {origin_locode}")
logger.info(f"Found {len(dest_nodes)} sectionjunction nodes for {dest_locode}")

## 4. Reference Data: Zenodo Geocoded UN/LOCODEs
We use the geocoded dataset as a coordinate fallback for locations missing in the physical network.

In [ ]:
zenodo_url = (
    "https://zenodo.org/records/11191511/files/unlo-geocoded-v0.1.gpkg?download=1"
)
logger.info(f"Loading Zenodo reference data from {zenodo_url}...")

zenodo_gdf = gpd.read_file(zenodo_url)
logger.info(f"Loaded {len(zenodo_gdf)} geocoded locations.")

# Lookup Rotterdam
rtm_ref = zenodo_gdf[
    (zenodo_gdf["country_code"] == "NL") & (zenodo_gdf["location_code"] == "RTM")
]
assert not rtm_ref.empty, "Rotterdam (NLRTM) not found in Zenodo data"
logger.info(
    f"Zenodo Reference for Rotterdam: {rtm_ref.iloc[0]['name']} at {rtm_ref.geometry.iloc[0]}"
)

## 5. Route and Visualize
Compute the shortest path between the first matching sectionjunction nodes.

In [ ]:
# Pick first matching node
start_node_id = origin_nodes.index[0]
end_node_id = dest_nodes.index[0]

logger.info(f"Computing path from {start_node_id} to {end_node_id}...")
route = nx.shortest_path(graph, start_node_id, end_node_id, weight="length_m")

# Extract geometries for visualization
geoms = []
for u, v in itertools.pairwise(route):
    edge_data = graph.edges[u, v]
    geom = edge_data.get("geometry")
    if isinstance(geom, str):
        geom = shapely.wkt.loads(geom)
    if geom:
        geoms.append(geom)

route_gdf = gpd.GeoDataFrame(geometry=geoms)
assert not route_gdf.empty, "No geometries found for the computed route"

# Simple plot
fig, ax = plt.subplots(figsize=(10, 8))
nodes_gdf.plot(ax=ax, color="lightgrey", markersize=1, alpha=0.5)
route_gdf.plot(
    ax=ax,
    color="blue",
    linewidth=2,
    label=f"{origin_locode} to {dest_locode}",
)
ax.set_title(f"Cargo Route: {origin_locode} to {dest_locode}")
plt.legend()
plt.show()